In [5]:
pip install -r supply-chain-analytics/requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

'C:\\Users\\Matthew\\supply-chain-analytics\\notebooks'

In [5]:
raw_data_folder = "../data/raw/"
raw_data_path = raw_data_folder + "DataCoSupplyChainDataset.csv"

df = pd.read_csv(raw_data_path, encoding='cp1252')

In [6]:
# # of records
df.shape

(180519, 53)

In [7]:
# list columns
df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Product Category Id',
       'Product De

In [8]:
# missing values?
df.isnull().sum()[df.isnull().sum() > 0]

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

In [9]:
# how many deliver statuses?
df["Delivery Status"].value_counts()

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

In [10]:
# order statuses?
df["Order Status"].value_counts()

Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64

In [11]:
# shipping types
df["Shipping Mode"].value_counts()

Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

In [12]:
# delivery risk spread
df["Late_delivery_risk"].value_counts()

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

In [13]:
# create new df to clean
df_clean = df.copy()

In [14]:
# drop unnecessary columns
# 8 missing values for 'Customer Lname', investigate correlation to delivery delays before dropping
drop_cols = [
    'Customer Password',
    'Customer Email',
    'Customer Street'
    'Customer Fname',
    'Product Description'
]

In [15]:
# clean dates
df_clean['order date (DateOrders)'] = pd.to_datetime(
    df_clean['order date (DateOrders)']
)
df_clean['shipping date (DateOrders)'] = pd.to_datetime(
    df_clean['shipping date (DateOrders)']
)
df_clean["order_year"] = (
    df_clean["order date (DateOrders)"]
    .dt.year
)
df_clean["order_month"] = (
    df_clean["order date (DateOrders)"]
    .dt.month_name()
)
df_clean["order_quarter"] = (
    df_clean["order date (DateOrders)"]
    .dt.quarter
)
# raw
print(f"raw: {df['order date (DateOrders)'][1]}")
# cleaned
print(f"cleaned: {df_clean['shipping date (DateOrders)'][1]}")

raw: 1/13/2018 12:27
cleaned: 2018-01-18 12:27:00


In [16]:
# validate shipping days (takes at least 1 day to ship, both scheduled and actual)
df_clean[
    df_clean['Days for shipment (scheduled)'] < 0
]
df_clean[
    df_clean['Days for shipping (real)'] < 0
]

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,order_year,order_month,order_quarter


In [17]:
# add shipping variance column (Early: < 0, Late: > 0)
df_clean['shipping_variance'] = (
    df_clean['Days for shipping (real)']
    - df_clean['Days for shipment (scheduled)']
)

In [18]:
# get columns that use string values
string_cols = df_clean.select_dtypes(include=['string']).columns
# remove whitespaces
df_clean[string_cols] = df_clean[string_cols].apply(lambda x: x.str.strip())

In [25]:
# create late, on-time, and early delivery flags for later use

# late
df_clean["late_delivery_flag"] = (
    df_clean["shipping_variance"] > 0
).astype(int)

# early
df_clean["early_delivery_flag"] = (
    df_clean["shipping_variance"] < 0
).astype(int)

# on-time
df_clean["ontime_delivery_flag"] = (
    df_clean["shipping_variance"] == 0
).astype(int)

In [29]:
df_clean.to_csv(r'../data/cleaned/cleaned_data.csv', index=False)